In [1]:
import sys

print(sys.executable)

c:\Users\lizcr\OneDrive\Documents\MSc\Project\msc_project\.venv\Scripts\python.exe


In [7]:
import copy, math, os, pickle, time, pandas as pd, numpy as np, scipy.stats as ss

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score, f1_score

import torch, torch.utils.data as utils, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.autograd import Variable
from torch.nn.parameter import Parameter

In [8]:
GAP_TIME          = 6  # In hours
WINDOW_SIZE       = 24 # In hours
SEED              = 1
ID_COLS           = ['subject_id', 'hadm_id', 'icustay_id']
id_cols_hourly = ID_COLS + ['hours_in']

np.random.seed(SEED)
torch.manual_seed(SEED)

In [1]:
class DictDist():
    def __init__(self, dict_of_rvs): self.dict_of_rvs = dict_of_rvs
    def rvs(self, n):
        a = {k: v.rvs(n) for k, v in self.dict_of_rvs.items()}
        out = []
        for i in range(n): out.append({k: vs[i] for k, vs in a.items()})
        return out
    
class Choice():
    def __init__(self, options): self.options = options
    def rvs(self, n): return [self.options[i] for i in ss.randint(0, len(self.options)).rvs(n)]

In [2]:
import getpass
from sqlalchemy import create_engine

pg_user = 'postgres'      # same value you use to connect via psql
pg_host = 'localhost'          # or wherever your Postgres server is
pg_port = 5432
pg_dbname = 'mimiciv'

pg_password = getpass.getpass('Postgres password: ')

engine = create_engine(
    f'postgresql+psycopg2://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_dbname}'
)

In [4]:
import pandas as pd
df = pd.read_sql("SELECT current_database();", engine)
print(df)

  current_database
0          mimiciv


In [5]:
pd.read_sql('SELECT 1', engine)

,?column?
0,1


In [9]:
%%time

data_full_lvl2 = pd.read_sql('SELECT * FROM msc_project.hourly_data', engine)
data_full_lvl2 = data_full_lvl2.set_index(id_cols_hourly)

statics = pd.read_sql('SELECT * FROM msc_project.allpatients', engine)
statics = statics.set_index(ID_COLS)

CPU times: total: 1min 25s
Wall time: 6min 8s
